# Peaked-circuit MPS smoke test

Run the deterministic mirrored peaked family through Qiskit Aer MPS and MettleQ routed MPS with the same shots and known mode.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

A mirrored peaked circuit creates temporary entanglement and then concentrates probability on a known bitstring.

In [2]:
from qiskit_aer import AerSimulator
from mettleq.peaked import build_mirrored_peaked_circuit

circuit = build_mirrored_peaked_circuit(8, depth=2, topology="long_range", seed=153, measure=True)
peak = circuit.metadata["peak_bitstring"]
expected_probability = circuit.metadata["expected_peak_probability"]
shots = 4096
aer = AerSimulator(method="matrix_product_state")
aer_circuit = transpile(circuit, aer, optimization_level=1)

def run_reference():
    return aer.run(aer_circuit, shots=shots, seed_simulator=153).result().get_counts()

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(run_reference)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
)
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    return backend.run(compiled, shots=shots, seed_simulator=153).result().get_counts()

candidate, mettleq_ms, _ = benchmark(run_mettleq)
tvd = total_variation_distance(reference, candidate)
observed = candidate.get(peak, 0) / shots
mode = max(candidate, key=candidate.get)
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

The expected mode, peak probability, and total-variation distance must all pass; this is stronger than checking only runtime.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/16_peaked_circuit_smoke.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="expected mode, peak probability atol=0.025, and TVD<=0.06",
    passed=mode == peak and abs(observed - expected_probability) <= 0.025 and tvd <= 0.06,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"tvd": tvd, "expected_peak": peak, "mettleq_mode": mode, "expected_probability": expected_probability, "observed_probability": observed},
    notes="This mirrored regression is not the full 56-qubit P9 quantum-advantage instance.",
)


Comparison summary
------------------
Correctness contract: PASS — expected mode, peak probability atol=0.025, and TVD<=0.06
SDK reference median: 9.365 ms
MettleQ median:       36.335 ms
Timing interpretation: the SDK reference was 3.880x faster in this run.
MettleQ selected: matrix_product_state / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)
Note: This mirrored regression is not the full 56-qubit P9 quantum-advantage instance.

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "expected mode, peak probability atol=0.025, and TVD<=0.06", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"expected_peak": "01001100", "expected_probability": 0.9500366589899866, "mettleq_mode": "01001100", "observed_probability": 0.955078125, "tvd": 0.0048828125}, "mettleq_median_ms": 36.33541698218323, "notebook": "qiskit/16_peaked_circuit_smoke.ipynb", "notes": "This mirrored regression is not the full 56-qu

## What should you conclude?

Use it as a quick MPS regression. The full 56-qubit P9 benchmark uses the separate midpoint-MPO method.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.